# Restaurant Review Sentiment Analysis Using Machine Learning

**University:** ITM Skills University  
**Program:** B.Tech CSE 2024–28  
**Subject:** Machine Learning  
**Case Study:** Case Study No. 77 – Sentiment Analysis  
**Project Title:** "Restaurant Review Sentiment Analysis Using Machine Learning"  
**Machine Learning Type:** Supervised Machine Learning (Multiclass / Text Classification)  

---
### Project Architecture & Workflow
```
Customer Review Text -> Text Cleaning & Preprocessing -> TF-IDF Vectorization -> ML Model -> Sentiment Prediction (Positive / Negative)
```

## 1. Import Libraries
We import standard, beginner-friendly Python libraries for data processing (`pandas`, `numpy`), visualization (`matplotlib`, `seaborn`), NLP feature extraction and machine learning (`scikit-learn`), and model persistence (`joblib`).

In [ ]:
import re
import string
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# Plot style setting
sns.set_theme(style="whitegrid")
print("All libraries imported successfully!")

## 2. Load Dataset
We load the genuine restaurant reviews dataset from `dataset.csv`.

In [ ]:
df = pd.read_csv("dataset.csv")
df.head(5)

## 3. Dataset Overview
Inspect the dataset shape, columns, data types, missing values, duplicates, and initial class balance.

In [ ]:
print("=== DATASET OVERVIEW ===")
print(f"Total Rows: {df.shape[0]}")
print(f"Total Columns: {df.shape[1]}")
print("\n--- Column Names & Data Types ---")
print(df.dtypes)
print("\n--- Missing Values ---")
print(df.isnull().sum())
print(f"\nDuplicate Rows: {df.duplicated().sum()}")
print("\n--- Sentiment Class Distribution ---")
print(df["Sentiment"].value_counts())

## 4. Data Cleaning
We clean the dataset by removing duplicate reviews and verifying that there are no null values.

In [ ]:
initial_count = len(df)
df = df.drop_duplicates(subset=["Review"]).dropna(subset=["Review", "Sentiment"]).reset_index(drop=True)
print(f"Initial records: {initial_count}")
print(f"Records after dropping duplicates: {len(df)}")
print(f"Cleaned sentiment counts:\n{df['Sentiment'].value_counts()}")

## 5. Exploratory Data Analysis (EDA)
We create clear visualizations to inspect the sentiment distribution and word count distribution across reviews.

In [ ]:
# Graph 1: Sentiment Distribution
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x="Sentiment", palette=["#2ecc71", "#e74c3c"])
plt.title("Sentiment Distribution in Dataset", fontsize=13, fontweight="bold")
plt.xlabel("Sentiment", fontsize=11)
plt.ylabel("Number of Reviews", fontsize=11)
plt.show()
print("Observation: The dataset is balanced with 499 Positive and 497 Negative reviews, ensuring fair training without class bias.")

In [ ]:
# Graph 2: Review Word Count Distribution
df["Word_Count"] = df["Review"].apply(lambda x: len(str(x).split()))

plt.figure(figsize=(8, 4))
sns.histplot(data=df, x="Word_Count", hue="Sentiment", bins=25, kde=True, palette=["#2ecc71", "#e74c3c"])
plt.title("Review Word Count Distribution by Sentiment", fontsize=13, fontweight="bold")
plt.xlabel("Word Count (Number of Words)", fontsize=11)
plt.ylabel("Frequency", fontsize=11)
plt.show()
print("Observation: Most reviews are concise (5 to 25 words). Positive and negative reviews have very similar length profiles.")

## 6. Text Preprocessing
We clean raw customer text reviews by:
1. Converting all characters to lowercase.
2. Removing URLs or links.
3. Removing punctuation, numbers, and special characters.
4. Removing extra whitespace.

In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+|https\S+", "", text, flags=re.MULTILINE)
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["Cleaned_Review"] = df["Review"].apply(clean_text)
df[["Review", "Cleaned_Review", "Sentiment"]].head(5)

## 7. Train/Test Split
We split the dataset into 80% Training set (796 reviews) and 20% Testing set (200 reviews) using `random_state=42` and stratification.

In [ ]:
X = df["Cleaned_Review"]
y = df["Sentiment"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Training set samples: {len(X_train)} (80%)")
print(f"Testing set samples:  {len(X_test)} (20%)")
print("\nTest class distribution:\n", y_test.value_counts())

## 8. TF-IDF Vectorization
**What is TF-IDF?**
- **TF (Term Frequency):** Frequency of a word in a specific review.
- **IDF (Inverse Document Frequency):** Rarity/importance of a word across the entire dataset.
- **TF-IDF = TF × IDF:** Converts raw text into numerical feature vectors.
- We fit the vectorizer **only on the training set** to avoid data leakage.

In [ ]:
vectorizer = TfidfVectorizer()
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print(f"Vocabulary Size: {len(vectorizer.vocabulary_)} unique words")
print(f"Training TF-IDF matrix shape: {X_train_tfidf.shape}")
print(f"Testing TF-IDF matrix shape:  {X_test_tfidf.shape}")

## 9. Model 1: Logistic Regression
Logistic Regression is a linear supervised classification algorithm that uses the sigmoid function to calculate sentiment probability.

In [ ]:
lr_model = LogisticRegression(random_state=42)
lr_model.fit(X_train_tfidf, y_train)
lr_preds = lr_model.predict(X_test_tfidf)
print("Logistic Regression model trained successfully!")

## 10. Model 2: Decision Tree Classifier
Decision Tree is a tree-structured supervised classifier that recursively partitions the feature space based on word presence thresholds.

In [ ]:
dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train_tfidf, y_train)
dt_preds = dt_model.predict(X_test_tfidf)
print("Decision Tree model trained successfully!")

## 11. Model Comparison
We compare the actual performance metrics of both models on the unseen test set.

In [ ]:
# Metrics for Logistic Regression
lr_acc = accuracy_score(y_test, lr_preds)
lr_prec = precision_score(y_test, lr_preds, pos_label="Positive")
lr_rec = recall_score(y_test, lr_preds, pos_label="Positive")
lr_f1 = f1_score(y_test, lr_preds, pos_label="Positive")

# Metrics for Decision Tree Classifier
dt_acc = accuracy_score(y_test, dt_preds)
dt_prec = precision_score(y_test, dt_preds, pos_label="Positive")
dt_rec = recall_score(y_test, dt_preds, pos_label="Positive")
dt_f1 = f1_score(y_test, dt_preds, pos_label="Positive")

comparison_df = pd.DataFrame({
    "Model": ["Logistic Regression", "Decision Tree Classifier"],
    "Accuracy": [lr_acc, dt_acc],
    "Precision": [lr_prec, dt_prec],
    "Recall": [lr_rec, dt_rec],
    "F1 Score": [lr_f1, dt_f1]
})

print("=== ACTUAL MODEL COMPARISON TABLE ===")
print(comparison_df.to_string(index=False))

# Visual Comparison Chart
comparison_melted = comparison_df.melt(id_vars="Model", var_name="Metric", value_name="Score")
plt.figure(figsize=(8, 4))
sns.barplot(data=comparison_melted, x="Metric", y="Score", hue="Model", palette=["#3498db", "#e67e22"])
plt.title("Model Comparison: Logistic Regression vs Decision Tree", fontsize=13, fontweight="bold")
plt.ylim(0.55, 0.95)
plt.ylabel("Score", fontsize=11)
plt.show()

## 12. Evaluation of Selected Model
Detailed evaluation metrics and classification report for Logistic Regression on unseen test data.

In [ ]:
print("=== CLASSIFICATION REPORT (Logistic Regression) ===")
print(classification_report(y_test, lr_preds))
print(f"Accuracy  : {lr_acc:.4f} ({lr_acc*100:.2f}%)")
print(f"Precision : {lr_prec:.4f}")
print(f"Recall    : {lr_rec:.4f}")
print(f"F1 Score  : {lr_f1:.4f}")

## 13. Confusion Matrix
Visualizing True Positives (TP), True Negatives (TN), False Positives (FP), and False Negatives (FN).

In [ ]:
cm = confusion_matrix(y_test, lr_preds, labels=["Negative", "Positive"])

plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Predicted Negative", "Predicted Positive"],
            yticklabels=["Actual Negative", "Actual Positive"])
plt.title("Confusion Matrix (Logistic Regression)", fontsize=13, fontweight="bold")
plt.xlabel("Predicted Sentiment", fontsize=11)
plt.ylabel("Actual Sentiment", fontsize=11)
plt.show()

print(f"True Negatives (TN): {cm[0, 0]} | False Positives (FP): {cm[0, 1]}")
print(f"False Negatives (FN): {cm[1, 0]} | True Positives (TP): {cm[1, 1]}")

## 14. Error Analysis
We inspect genuine misclassified test reviews to understand why the model made mistakes (e.g. subtle sarcasm, negation, or short phrasing).

In [ ]:
test_results = pd.DataFrame({
    "Review": df.loc[X_test.index, "Review"].values,
    "Actual Sentiment": y_test.values,
    "Predicted Sentiment": lr_preds
})
errors = test_results[test_results["Actual Sentiment"] != test_results["Predicted Sentiment"]]
print(f"Total Misclassified Test Reviews: {len(errors)} out of {len(test_results)}\n")
display_errors = errors.head(10)
for idx, row in display_errors.iterrows():
    print(f'Review: "{row["Review"]}"')
    print(f'Actual: {row["Actual Sentiment"]} | Predicted: {row["Predicted Sentiment"]}')
    print('-' * 60)

## 15. Select Best Model
**Selected Model:** Logistic Regression  
**Reason:** Logistic Regression achieved an accuracy of **83.00%**, high precision (**81.73%**), and well-balanced classification across both positive and negative sentiment classes without bias.

In [ ]:
best_model = lr_model
print("Selected Best Model: Logistic Regression")

## 16. Save Model
We serialize and export the trained model (`model.pkl`) and TF-IDF vectorizer (`vectorizer.pkl`) using `joblib` so that our Streamlit application can perform inference instantly.

In [ ]:
joblib.dump(best_model, "model.pkl")
joblib.dump(vectorizer, "vectorizer.pkl")
print("Saved model -> model.pkl")
print("Saved vectorizer -> vectorizer.pkl")

## 17. Final Conclusion
- We successfully designed and implemented an end-to-end supervised sentiment classification pipeline.
- Text Preprocessing + TF-IDF Vectorization + Logistic Regression achieved **83.00% accuracy** on unseen test reviews.
- The model artifacts (`model.pkl` and `vectorizer.pkl`) are saved and ready for the Streamlit web application.